# KOVA Visual OCR — Google Colab GPU Worker

Worker này chỉ chạy **OCR chữ/phụ đề đã hiển thị trong khung hình**, độc lập với notebook Speech-to-Text và KOVA Voice Studio.

1. Chọn **Runtime → Change runtime type → GPU**.
2. Chọn **Runtime → Run all**.
3. Dán `KOVA_OCR_URL` và `KOVA_OCR_TOKEN` vào KOVA → **Nguồn video** → chọn **OCR chữ/phụ đề có sẵn** → **Google Colab GPU**.

KOVA tải video về desktop trước rồi gửi video cùng vùng OCR (ROI) sang worker GPU này. URL/token chỉ tồn tại đến khi phiên Colab hoặc tunnel dừng.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/khoinguyen59/kova-video-dubbing.git'
WORKSPACE = Path('/content/kova-video-dubbing')

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

run(['nvidia-smi'])
# Colab currently uses Python 3.12. Install Paddle from its CUDA wheel index,
# not generic PyPI resolution (which can select an incompatible or CPU wheel).
run([sys.executable, '-m', 'pip', 'uninstall', '--yes', 'paddlepaddle', 'paddlepaddle-gpu', 'paddleocr'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', '--no-cache-dir',
     'paddlepaddle-gpu==3.3.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--no-cache-dir',
     'fastapi==0.115.12', 'uvicorn[standard]==0.34.2', 'python-multipart==0.0.20',
     'opencv-python-headless', 'paddleocr==3.3.3'])
if WORKSPACE.exists():
    run(['git', 'fetch', 'origin', 'main'], cwd=WORKSPACE)
    run(['git', 'reset', '--hard', 'origin/main'], cwd=WORKSPACE)
else:
    run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(WORKSPACE)])
BRIDGE = WORKSPACE / 'scripts' / 'kova_visual_ocr.py'
if not BRIDGE.is_file():
    raise RuntimeError(f'Missing KOVA OCR bridge: {BRIDGE}. Publish the current KOVA source to GitHub first.')
GPU_CHECK = "import json, paddle; assert paddle.is_compiled_with_cuda(), 'Installed Paddle wheel has no CUDA support'; assert paddle.device.cuda.device_count() > 0, 'No CUDA device is visible; choose a GPU runtime'; print(json.dumps({'paddle': paddle.__version__, 'gpu_count': paddle.device.cuda.device_count()}))"
gpu_check = subprocess.run([sys.executable, '-c', GPU_CHECK], text=True, capture_output=True)
if gpu_check.returncode:
    raise RuntimeError('Paddle CUDA check failed.\nstdout:\n' + gpu_check.stdout[-3000:] + '\nstderr:\n' + gpu_check.stderr[-3000:])
preflight = subprocess.run([sys.executable, str(BRIDGE), '--preflight'], text=True, capture_output=True)
if preflight.returncode:
    raise RuntimeError('KOVA OCR bridge preflight failed.\nstdout:\n' + preflight.stdout[-3000:] + '\nstderr:\n' + preflight.stderr[-3000:])
print('Installed OCR dependencies and prepared bridge:', BRIDGE, gpu_check.stdout.strip(), preflight.stdout.strip())

In [ ]:
from pathlib import Path

WORKER = Path('/content/kova_ocr_worker.py')
WORKER.write_text(r'''
import json
import os
import secrets
import shutil
import subprocess
import sys
from pathlib import Path

import paddle
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile

TOKEN = os.environ['KOVA_OCR_API_TOKEN']
BRIDGE = Path(os.environ['KOVA_OCR_BRIDGE'])
UPLOAD_DIR = Path('/content/kova-ocr-uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if not paddle.is_compiled_with_cuda():
    raise RuntimeError('PaddlePaddle CUDA is unavailable. Stop and select a GPU runtime; CPU fallback is disabled.')

app = FastAPI(title='KOVA Colab Visual OCR Worker', docs_url=None, redoc_url=None, openapi_url=None)

def authorize(authorization: str = Header(default='')):
    if not secrets.compare_digest(authorization, 'Bearer ' + TOKEN):
        raise HTTPException(status_code=401, detail='invalid or missing bearer token')

@app.get('/health')
def health(_: None = Depends(authorize)):
    return {'ready': True, 'device': 'cuda', 'engine': 'PaddleOCR'}

@app.post('/v1/ocr/extract')
async def extract(
    file: UploadFile = File(...),
    roi: str = Form(...),
    language: str = Form(default='en'),
    interval_ms: int = Form(default=250),
    merge_gap_ms: int = Form(default=450),
    _: None = Depends(authorize),
):
    suffix = Path(file.filename or 'source.mp4').suffix or '.mp4'
    request_dir = UPLOAD_DIR / secrets.token_urlsafe(16)
    request_dir.mkdir(parents=True, exist_ok=True)
    input_path = request_dir / ('source' + suffix)
    output_path = request_dir / 'origin_language.srt'
    try:
        with input_path.open('wb') as output:
            shutil.copyfileobj(file.file, output)
        process = subprocess.run([
            sys.executable, str(BRIDGE), '--input', str(input_path), '--output', str(output_path),
            '--roi', roi, '--lang', language, '--device', 'gpu',
            '--interval-ms', str(interval_ms), '--merge-gap-ms', str(merge_gap_ms),
        ], capture_output=True, text=True, timeout=3300, check=False)
        if process.returncode != 0 or not output_path.is_file():
            detail = (process.stderr or process.stdout)[-3000:]
            raise HTTPException(status_code=503, detail='PaddleOCR GPU extraction failed: ' + detail)
        result = {}
        for line in reversed(process.stdout.splitlines()):
            try:
                result = json.loads(line)
                break
            except json.JSONDecodeError:
                pass
        return {
            'srt': output_path.read_text(encoding='utf-8'),
            'device': 'cuda',
            'frame_count': int(result.get('frame_count', 0)),
            'cue_count': int(result.get('cue_count', 0)),
            'dropped_frames': int(result.get('dropped_frames', 0)),
            'normalized_cjk': bool(result.get('normalized_cjk', False)),
        }
    finally:
        await file.close()
        shutil.rmtree(request_dir, ignore_errors=True)
''', encoding='utf-8')

print('OCR worker source and CUDA/Paddle dependency check passed.')

In [ ]:
import json, secrets, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
ENV = os.environ.copy()
ENV.update({'KOVA_OCR_API_TOKEN': TOKEN, 'KOVA_OCR_BRIDGE': str(BRIDGE)})
LOG = '/content/kova-ocr-worker.log'
subprocess.run(['pkill', '-f', 'kova_ocr_worker:app'], check=False)
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'kova_ocr_worker:app', '--host', '127.0.0.1', '--port', '3950'], cwd='/content', env=ENV, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        request = urllib.request.Request('http://127.0.0.1:3950/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=5) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('OCR worker did not become CUDA-ready. Log tail:\n' + Path(LOG).read_text(errors='replace')[-4000:])

run(['wget', '-q', '-O', '/content/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared.deb'])
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3950', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    if 'https://' in line and 'trycloudflare.com' in line:
        public_url = line[line.find('https://'):].split()[0]
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found.')

print('\nKOVA_OCR_URL=' + public_url)
print('KOVA_OCR_TOKEN=' + TOKEN)
print('ENGINE=PaddleOCR; DEVICE=cuda')
print('Paste this URL and token into KOVA > Nguồn video > OCR > Google Colab GPU. Do not append /v1.')